<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/04_Foundations/mathematics/03_calculus_gradient_descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Calculus & Gradient Descent

Gradient descent is the engine of machine learning. Every neural network, every optimization algorithm, every model update is calculus in action. This notebook builds intuition from first principles.

**Topics:**
- Derivatives and the gradient
- Partial derivatives and the Jacobian
- Chain rule (backpropagation is just chain rule)
- Gradient descent: vanilla, SGD, momentum
- Adaptive optimizers: AdaGrad, RMSProp, Adam
- Visualizing optimization landscapes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

np.random.seed(42)

## 1. Derivatives — The Rate of Change

In [ ]:
# Numerical differentiation: f'(x) ≈ [f(x+h) - f(x-h)] / 2h
def numerical_gradient(f, x, h=1e-5):
    """Central difference approximation of gradient."""
    grad = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += h
        x_minus = x.copy(); x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad

# Example: f(x) = x^4 - 3x^3 + 2  →  f'(x) = 4x^3 - 9x^2
f = lambda x: x[0]**4 - 3*x[0]**3 + 2
f_prime_analytical = lambda x: 4*x[0]**3 - 9*x[0]**2

x_test = np.array([2.0])
numerical = numerical_gradient(f, x_test)
analytical = f_prime_analytical(x_test)

print(f'At x=2: numerical f\'(x) = {numerical[0]:.6f}')
print(f'        analytical f\'(x) = {analytical:.6f}')
print(f'        error = {abs(numerical[0] - analytical):.2e}')

# Plot function and its derivative
x = np.linspace(-1, 4, 300)
y = x**4 - 3*x**3 + 2
dy = 4*x**3 - 9*x**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(x, y, 'b-', lw=2, label='f(x) = x⁴ - 3x³ + 2')
ax1.axhline(0, color='k', lw=0.5); ax1.axvline(0, color='k', lw=0.5)
ax1.set_title('f(x)'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(x, dy, 'r-', lw=2, label="f'(x) = 4x³ - 9x²")
ax2.axhline(0, color='k', lw=0.5); ax2.axvline(0, color='k', lw=0.5)
# Critical points where f'(x) = 0
ax2.scatter([0, 2.25], [0, 0], color='green', zorder=5, s=100, label='Critical points (f\'=0)')
ax2.set_title("f'(x)"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 2. Gradient — Multi-Dimensional Derivative

For f(x, y), the gradient **∇f = [∂f/∂x, ∂f/∂y]** points in the direction of steepest ascent.

In [ ]:
# 2D loss surface: f(x,y) = (x-1)^2 + 2(y-2)^2  (bowl-shaped)
def loss(params):
    x, y = params
    return (x - 1)**2 + 2*(y - 2)**2

def grad_loss(params):
    x, y = params
    return np.array([2*(x-1), 4*(y-2)])

# Visualize gradient field
x_vals = np.linspace(-2, 4, 100)
y_vals = np.linspace(-1, 5, 100)
X, Y = np.meshgrid(x_vals, y_vals)
Z = (X - 1)**2 + 2*(Y - 2)**2

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)
plt.colorbar(contour, ax=ax)

# Gradient arrows (negative gradient = direction of descent)
gx_vals = np.linspace(-1.5, 3.5, 10)
gy_vals = np.linspace(-0.5, 4.5, 10)
GX, GY = np.meshgrid(gx_vals, gy_vals)
U = -2*(GX - 1)
V = -4*(GY - 2)
ax.quiver(GX, GY, U, V, alpha=0.6, color='white', scale=50)

ax.scatter(1, 2, color='red', s=200, zorder=5, marker='*', label='Minimum (1, 2)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Loss Surface with Negative Gradient Field (direction of descent)')
ax.legend()
plt.tight_layout(); plt.show()

## 3. Chain Rule — Backpropagation's Foundation

For composed functions: **d/dx f(g(x)) = f'(g(x)) · g'(x)**

In [ ]:
# Manual backpropagation of a 2-layer computation
# Forward pass: y = sigmoid(w2 * relu(w1 * x + b1) + b2)

def relu(z): return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu_grad(z): return (z > 0).astype(float)
def sigmoid_grad(z): 
    s = sigmoid(z)
    return s * (1 - s)

# Parameters
np.random.seed(42)
w1, b1 = 0.5, -0.3
w2, b2 = 0.8, 0.1
x_in, y_true = 2.0, 1.0
lr = 0.1

losses = []
for _ in range(50):
    # Forward pass
    z1 = w1 * x_in + b1
    a1 = relu(z1)
    z2 = w2 * a1 + b2
    y_hat = sigmoid(z2)
    loss_val = 0.5 * (y_hat - y_true)**2
    losses.append(loss_val)

    # Backward pass (chain rule)
    dL_dyhat = y_hat - y_true          # dL/dŷ
    dyhat_dz2 = sigmoid_grad(z2)       # dŷ/dz2
    dz2_dw2 = a1                       # dz2/dw2
    dz2_db2 = 1.0                      # dz2/db2
    dz2_da1 = w2                       # dz2/da1
    da1_dz1 = relu_grad(z1)            # da1/dz1
    dz1_dw1 = x_in                     # dz1/dw1
    dz1_db1 = 1.0                      # dz1/db1

    # Chain rule: multiply gradients along path
    delta = dL_dyhat * dyhat_dz2
    dL_dw2 = delta * dz2_dw2
    dL_db2 = delta * dz2_db2
    dL_dw1 = delta * dz2_da1 * da1_dz1 * dz1_dw1
    dL_db1 = delta * dz2_da1 * da1_dz1 * dz1_db1

    # Update
    w1 -= lr * dL_dw1
    b1 -= lr * dL_db1
    w2 -= lr * dL_dw2
    b2 -= lr * dL_db2

plt.figure(figsize=(8, 4))
plt.plot(losses, 'b-o', markersize=4)
plt.xlabel('Iteration'); plt.ylabel('Loss')
plt.title('Loss Decreasing via Backpropagation + Gradient Descent')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final loss: {losses[-1]:.6f}, Final prediction: {sigmoid(w2 * relu(w1 * x_in + b1) + b2):.4f} (target: {y_true})')

## 4. Gradient Descent Variants

In [ ]:
# Compare optimizers on the Rosenbrock "banana" function
# f(x,y) = (1-x)^2 + 100(y-x^2)^2  — minimum at (1,1), notoriously hard

def rosenbrock(params):
    x, y = params
    return (1 - x)**2 + 100*(y - x**2)**2

def rosenbrock_grad(params):
    x, y = params
    dx = -2*(1-x) + 100*2*(y - x**2)*(-2*x)
    dy = 100*2*(y - x**2)
    return np.array([dx, dy])

def run_optimizer(optimizer_fn, start=(-1.5, 0.5), steps=2000):
    """Run an optimizer and return the trajectory."""
    path = [np.array(start, dtype=float)]
    state = optimizer_fn(np.array(start, dtype=float))
    p = np.array(start, dtype=float)
    opt = optimizer_fn(p)
    trajectory = [p.copy()]
    for _ in range(steps):
        g = rosenbrock_grad(p)
        p = opt(g)
        trajectory.append(p.copy())
    return np.array(trajectory)

# Vanilla gradient descent
def vanilla_gd(p0, lr=0.001):
    p = p0.copy()
    def step(g):
        nonlocal p
        p = p - lr * g
        return p.copy()
    return step

# Momentum
def momentum_gd(p0, lr=0.001, beta=0.9):
    p = p0.copy()
    v = np.zeros_like(p0)
    def step(g):
        nonlocal p, v
        v = beta * v + (1 - beta) * g
        p = p - lr * v
        return p.copy()
    return step

# Adam optimizer
def adam_gd(p0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
    p = p0.copy()
    m, v = np.zeros_like(p0), np.zeros_like(p0)
    t = [0]
    def step(g):
        nonlocal p, m, v
        t[0] += 1
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t[0])
        v_hat = v / (1 - beta2**t[0])
        p = p - lr * m_hat / (np.sqrt(v_hat) + eps)
        return p.copy()
    return step

start = np.array([-1.5, 0.5])
steps = 1000

traj_vanilla = [start.copy()]
traj_momentum = [start.copy()]
traj_adam = [start.copy()]

p_v = start.copy(); opt_v = vanilla_gd(p_v, lr=0.001)
p_m = start.copy(); opt_m = momentum_gd(p_m, lr=0.001)
p_a = start.copy(); opt_a = adam_gd(p_a, lr=0.01)

for _ in range(steps):
    g_v = rosenbrock_grad(p_v); p_v = opt_v(g_v); traj_vanilla.append(p_v.copy())
    g_m = rosenbrock_grad(p_m); p_m = opt_m(g_m); traj_momentum.append(p_m.copy())
    g_a = rosenbrock_grad(p_a); p_a = opt_a(g_a); traj_adam.append(p_a.copy())

traj_vanilla = np.array(traj_vanilla)
traj_momentum = np.array(traj_momentum)
traj_adam = np.array(traj_adam)

# Plot
x_r = np.linspace(-2, 2, 200)
y_r = np.linspace(-1, 3, 200)
X_r, Y_r = np.meshgrid(x_r, y_r)
Z_r = (1 - X_r)**2 + 100*(Y_r - X_r**2)**2

fig, ax = plt.subplots(figsize=(10, 7))
ax.contourf(X_r, Y_r, np.log1p(Z_r), levels=30, cmap='RdYlBu_r', alpha=0.7)
ax.contour(X_r, Y_r, np.log1p(Z_r), levels=30, colors='white', alpha=0.2, linewidths=0.5)

ax.plot(traj_vanilla[:, 0], traj_vanilla[:, 1], 'b-', alpha=0.8, lw=1.5, label='Vanilla GD')
ax.plot(traj_momentum[:, 0], traj_momentum[:, 1], 'g-', alpha=0.8, lw=1.5, label='Momentum')
ax.plot(traj_adam[:, 0], traj_adam[:, 1], 'r-', alpha=0.8, lw=1.5, label='Adam')

ax.scatter(1, 1, color='yellow', s=200, zorder=6, marker='*', label='Global min (1,1)')
ax.scatter(*start, color='white', s=100, zorder=6, marker='o', label='Start')

ax.set_xlim(-2, 2); ax.set_ylim(-1, 3)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'Rosenbrock Function — Optimizer Comparison ({steps} steps)')
ax.legend()
plt.tight_layout(); plt.show()

print(f'Vanilla GD final: {traj_vanilla[-1].round(3)}, loss: {rosenbrock(traj_vanilla[-1]):.4f}')
print(f'Momentum   final: {traj_momentum[-1].round(3)}, loss: {rosenbrock(traj_momentum[-1]):.4f}')
print(f'Adam       final: {traj_adam[-1].round(3)}, loss: {rosenbrock(traj_adam[-1]):.4f}')

## 5. Learning Rate Effects

In [ ]:
# Simple quadratic: f(x) = x^2, optimum at x=0
f_simple = lambda x: x**2
grad_simple = lambda x: 2*x

lrs = [0.05, 0.5, 0.9, 1.1]  # stable, good, oscillating, diverging
colors_lr = ['green', 'steelblue', 'darkorange', 'red']
labels_lr = ['0.05 (too slow)', '0.5 (good)', '0.9 (oscillates)', '1.1 (diverges!)']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_plot = np.linspace(-4, 4, 200)
ax1.plot(x_plot, x_plot**2, 'k-', lw=2, label='f(x)=x²', alpha=0.5)

for lr, color, label in zip(lrs, colors_lr, labels_lr):
    x = 3.0
    traj = [x]
    for _ in range(20):
        x = x - lr * grad_simple(x)
        if abs(x) > 100: break
        traj.append(x)
    traj = np.array(traj)
    ax2.plot(traj, [f_simple(xi) for xi in traj], 'o-', color=color, lw=1.5, 
             markersize=4, label=f'lr={label}')

ax2.set_ylim(-0.5, 10); ax2.set_xlabel('Iteration')
ax2.set_ylabel('f(x)'); ax2.set_title('Effect of Learning Rate on Convergence')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

ax1.set_title('f(x) = x²'); ax1.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Summary

| Concept | Formula | ML Use |
|---------|---------|--------|
| Gradient | ∇f = [∂f/∂x₁, ..., ∂f/∂xₙ] | Direction of steepest ascent |
| Chain rule | d/dx f(g(x)) = f'(g(x))·g'(x) | Backpropagation |
| Gradient descent | θ = θ - lr·∇L(θ) | All ML optimization |
| Momentum | v = βv + (1-β)g; θ = θ - lr·v | Faster convergence |
| Adam | Adaptive per-parameter lr | Default optimizer for deep learning |

**Next:** [04_probability_basics.ipynb](04_probability_basics.ipynb)